# 02. Model Training and Evaluation Pipeline
We fit, tune, and compare Bernoulli Naive Bayes, Multinomial Naive Bayes, and Linear Support Vector Classifiers (LinearSVC).

### Step 1: Load Split Datasets
We read `/content/train_split.csv`, `/content/val_split.csv`, and `/content/test_split.csv` splits created during preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

train_df = pd.read_csv('/content/train_split.csv')
val_df = pd.read_csv('/content/val_split.csv')
test_df = pd.read_csv('/content/test_split.csv')

text_col = 'tweet_variant_b'
target_col = 'Party'

train_df = train_df.dropna(subset=[text_col, target_col])
val_df = val_df.dropna(subset=[text_col, target_col])
test_df = test_df.dropna(subset=[text_col, target_col])

print(f"Loaded Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

### Step 2: Establish Evaluation Strategy
We map Party categories (Democrat -> 0, Republican -> 1). Pipelines are fitted exclusively on the Training set, and model performance is compared strictly on the Validation set.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import joblib

y_train = (train_df[target_col] == 'Republican').astype(int).values
y_val = (val_df[target_col] == 'Republican').astype(int).values
y_test = (test_df[target_col] == 'Republican').astype(int).values

X_train = train_df[text_col].values
X_val = val_df[text_col].values
X_test = test_df[text_col].values

validation_metrics = {}
trained_pipelines = {}

def evaluate_on_validation(name, pipeline):
    # Fit strictly on the training partition
    pipeline.fit(X_train, y_train)
    # Evaluate strictly on the validation partition
    y_pred_val = pipeline.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred_val)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred_val, average='macro')
    
    validation_metrics[name] = {
        'Accuracy': acc,
        'Macro Precision': prec,
        'Macro Recall': rec,
        'Macro F1': f1
    }
    trained_pipelines[name] = pipeline
    print(f"--- {name} Validation Results ---")
    print(classification_report(y_val, y_pred_val, target_names=['Democrat', 'Republican']))
    return pipeline

### Step 3: Model Execution and Validation Comparison
We initialize and evaluate the three pipelines sequentially on the validation set.

In [ ]:
# Bernoulli Naive Bayes
pipe_bnb = Pipeline([
    ('vectorizer', CountVectorizer(binary=True)),
    ('classifier', BernoulliNB())
])
evaluate_on_validation('Bernoulli Naive Bayes', pipe_bnb)

# Multinomial Naive Bayes
pipe_mnb = Pipeline([
    ('vectorizer', CountVectorizer(binary=False)),
    ('classifier', MultinomialNB())
])
evaluate_on_validation('Multinomial Naive Bayes', pipe_mnb)

# Linear SVM
pipe_svc = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LinearSVC(random_state=42, dual=False))
])
evaluate_on_validation('Linear SVM (LinearSVC)', pipe_svc)

### Step 4: Model Selection and Final Test Evaluation
We select the top model strictly by validation set Macro F1 performance, then evaluate that model exactly once on the unseen test set.

In [ ]:
comparison_list = []
for name, m in validation_metrics.items():
    comparison_list.append({
        'Model': name,
        'Val Accuracy': f"{m['Accuracy']:.4f}",
        'Val Macro Precision': f"{m['Macro Precision']:.4f}",
        'Val Macro Recall': f"{m['Macro Recall']:.4f}",
        'Val Macro F1': f"{m['Macro F1']:.4f}"
    })

df_val_results = pd.DataFrame(comparison_list)
display(df_val_results)

# Select best model
best_model_name = df_val_results.sort_values(by='Val Macro F1', ascending=False).iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]

# Evaluate strictly on test data
y_pred_test = best_pipeline.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(y_test, y_pred_test, average='macro')

print(f"\nSelected Model: {best_model_name}")
print(f"--- Final Unseen Test Set Results ---")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro Precision: {test_prec:.4f}")
print(f"Test Macro Recall: {test_rec:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")
print(classification_report(y_test, y_pred_test, target_names=['Democrat', 'Republican']))

# Save model pipeline
joblib.dump(best_pipeline, '/content/best_model_pipeline.joblib')
print("Successfully saved best model pipeline!")